<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
M99 Skill Compliance
</b></font> </br></p>

---

In [ ]:
#@title Umgebung einrichten{ display-mode: "form" }
from pathlib import Path
import json
import subprocess
from textwrap import dedent

BASE = Path.cwd()
if BASE.name != 'Agenten':
    BASE = BASE / 'Agenten'

SKILL_DIR = BASE / '06_skill' / 'compliance'
NOTEBOOK_NAME = 'M99_skill_compliance.ipynb'
print('Notebook:', NOTEBOOK_NAME)
print('Skill-Verzeichnis:', SKILL_DIR)
print('Vorhanden:', SKILL_DIR.exists())


# 1 | Übersicht
---

Dieses Modul zeigt, warum **Skills** ein wichtiges Mittel zur Steuerung von Agenten sind.

Ein Skill ist kein normaler Prompt, sondern ein **wiederverwendbares Arbeitsrezept**:
- wann ein bestimmtes Vorgehen aktiviert wird
- welche Prüfschritte in welcher Reihenfolge nötig sind
- welche Regeln, Referenzen und Hilfsskripte der Agent heranzieht

Im Mittelpunkt steht hier ein **Compliance-Skill**. Er demonstriert drei Kernideen:

| Idee | Bedeutung im Agenten-Kontext |
|---|---|
| **Workflow-Orchestrierung** | Der Agent arbeitet feste Prüfschritte in Reihenfolge ab |
| **Guardrails** | Kritische Prüfungen werden nicht stillschweigend übersprungen |
| **Progressive Disclosure** | Nur die Kernlogik liegt in `SKILL.md`, Details werden bei Bedarf geladen |

**Lernziele:**
- Skills als Mittel zur Agentensteuerung einordnen
- eine gute Trennung zwischen `SKILL.md`, `references/` und `scripts/` verstehen
- einen Compliance-Skill lokal analysieren und testen


# 2 | Skill-Design für Agenten
---

<p><font color='black' size="5">Warum gehört das in einen Agenten-Kurs?</font></p>

Skills operationalisieren Fachwissen. Der Agent bekommt dadurch nicht nur Antworten, sondern eine **strukturierte Handlungslogik**.

Gerade bei Compliance-, Security- oder Freigabeprozessen reicht ein freier Prompt oft nicht aus. Man will stattdessen:
- Pflichtprüfungen definieren
- Eskalationsregeln festlegen
- Entscheidungsdokumentation erzwingen
- wiederkehrende Logik verlässlich wiederverwenden

<p><font color='black' size="5">Empfohlene Aufteilung</font></p>

| Datei-/Ordner-Typ | Rolle |
|---|---|
| `SKILL.md` | Kernablauf, Trigger, Guardrails, Verweise |
| `references/*.md` | Detailwissen, Regeln, Checklisten, Beispiele |
| `scripts/` | deterministische Hilfslogik für fragile oder wiederkehrende Schritte |

Das ist genau der Punkt, den Ihre ursprüngliche Frage adressiert hat: **Ja, Skills dürfen detailliert sein, aber die Details sollten sauber aufgeteilt werden.**

In [ ]:
for path in sorted(SKILL_DIR.rglob('*')):
    rel = path.relative_to(BASE)
    kind = 'DIR ' if path.is_dir() else 'FILE'
    print(f'{kind}  {rel}')


# 3 | Hands-On: Compliance-Skill lesen
---

In [ ]:
skill_text = (SKILL_DIR / 'SKILL.md').read_text(encoding='utf-8')
print(skill_text[:3000])


Achten Sie beim Lesen auf drei Dinge:
- **Trigger-Beschreibung:** Wann soll der Skill aktiv werden?
- **Core Workflow:** Welche Schritte sind verpflichtend?
- **Guardrails:** Was darf der Agent nicht behaupten oder überspringen?


In [ ]:
for name in ['checklist.md', 'risk_rules.md', 'examples.md']:
    path = SKILL_DIR / 'references' / name
    print(f'\n### {name}\n')
    print(path.read_text(encoding='utf-8')[:1500])


# 4 | Hands-On: deterministische Risiko-Prüfung
---

Der Skill enthält zusätzlich ein kleines Skript. Das ist nützlich, wenn ein Teil der Logik **nicht nur beschrieben, sondern zuverlässig berechnet** werden soll.

In [ ]:
case = {
    'subject_type': 'vendor',
    'subject_name': 'Example Trading LLC',
    'country': 'RU',
    'transaction_amount': 18000,
    'sanctions_hit': False,
    'adverse_media_hit': True,
    'pep_flag': False,
    'documents_complete': True,
}

cmd = [
    'python',
    str(SKILL_DIR / 'scripts' / 'assess_risk.py'),
    json.dumps(case),
]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(result.stdout)


In [ ]:
risk_result = json.loads(result.stdout)

decision_note = dedent(f'''
### Compliance Decision

- Case: {case['subject_type']} for {case['subject_name']}
- Checks performed: sanctions screening, geography review, transaction size review, documentation check
- Risk level: {risk_result['risk_level']}
- Decision: {risk_result['decision']}
- Rationale: {', '.join(risk_result['reasons'])}
- Missing information or escalation point: none in this training example
- Audit note: simulated training assessment, no live sanctions source connected
''').strip()

print(decision_note)


# 5 | Fazit
---

Dieses Beispiel zeigt die saubere Rollenverteilung eines Skills:
- `SKILL.md` steuert das Verhalten des Agenten
- `references/` halten Fachregeln und Beispiele aus dem Kernkontext heraus
- `scripts/` übernehmen deterministische Teilaufgaben

Genau deshalb passt das Thema in den Kurs **Agenten**: Es geht nicht nur um Antworten, sondern um **zuverlässig gesteuerte agentische Prozesse**.

# A | Aufgabe
---

<p><font color='black' size="5">
Skill erweitern
</font></p>

Erweitern Sie den Compliance-Skill um einen zweiten Entscheidungspfad, zum Beispiel für Lieferanten-Onboarding oder Auszahlungsfreigaben.

**Teilaufgaben:**
- Ergänzen Sie mindestens eine neue Regel in `references/risk_rules.md`
- Fügen Sie ein weiteres Beispiel in `references/examples.md` hinzu
- Passen Sie das Skript so an, dass die neue Regel in die Bewertung einfließt
